# Designingfor Reliability

**Module 12 · Lesson 12.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Reliability Math: Why Dependencies Erode Uptime
- Timeouts and Deadlines: Bound Every Wait
- Retry with Exponential Backoff and Full Jitter
- The Circuit Breaker: Stop Hammering a Dead Service
- Fallback and Graceful Degradation: Something Beats Nothing
- Bulkheads and Idempotent Retries: Contain and Deduplicate

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The 2am page: every user is getting a 500

> 💡 **Info**
>
> It is 2am and your phone lights up: every user of your AI app is getting a **500 error**. You check the logs - nothing in your code changed, no bad deploy, no traffic spike. What changed is that your **LLM provider is having a bad night**, and your app has no answer for that except to fail right along with it. This is the uncomfortable truth of production AI: your dependencies *will* rate-limit, time out, and go down, and the only thing you control is whether your app **degrades gracefully or falls over**. One tracker logged over a hundred LLM-provider incidents in a single quarter this year. This lesson builds the seven patterns that decide which side of that line you land on - starting with the math of *why* more dependencies means less uptime - and you can run every one of them with no API key, because reliability is engineering logic.

### 🎯 What you will be able to do after this lesson

- **Build** the core resilience patterns - a timeout, a retry with backoff and jitter, a circuit breaker, a fallback chain, and a bulkhead - as runnable code.

- **Compare** the reliability math of series dependencies (multiply down) with parallel redundancy (multiply up), and reason about an error budget.

- **Operate** each mechanism correctly: which errors to retry, when the breaker opens, how a deadline budget flows, and why retries need idempotency.

- **Evaluate** a composed resilient client and the order its layers must run in.

> 📦 **Info**
>
> ✅ Before you startThis deepens **12.1**’s provider fallback into a full reliability toolkit, and it builds on **11.9** (idempotency and the circuit breaker, now first-class here). Each pattern here is one you compose into 12.1’s reference architecture. The tooling around these patterns comes later: the AI gateway that applies them as config is **Lesson 12.3**, caching is **12.4**, and monitoring the error budget is **Lesson 12.8**.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🔧 **Analogy**
>
> Reliability is the difference between a **string of fairy lights** and a **house’s electrical wiring**. Cheap fairy lights are wired in series: one bulb dies and the whole string goes dark - that is a chain of dependencies, where any single failure takes everything down. A house is wired with **redundancy, fuses, and isolated circuits**: a blown fuse in the kitchen does not darken the bedroom, and a backup path keeps the essentials on. This lesson rewires your app from the fairy-light model to the house model. **Where it breaks down:** unlike a house, your dependencies fail constantly and unpredictably, so the redundancy has to be automatic - you cannot flip a breaker by hand at 2am.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“My app is as reliable as my provider.”** A chain of dependencies is *less* reliable than any one of them - availabilities multiply down. Only redundancy (a parallel fallback) buys uptime back.
> - **“If the provider is down, show an error.”** An error page is the worst outcome. Degrade instead - a cached, cheaper, or canned answer - because something beats nothing.

> 💡 **Info**
>
> ⚠️ Anti-patternRetrying **without jitter** and retrying the **wrong errors**. Without jitter, every client that failed at the same instant retries at the same instant - a thundering herd that re-crushes the service just as it recovers. And retrying a 401, a 400, or a context-overflow just burns attempts on an error that will never succeed. Retry only transient errors, with exponential backoff plus full jitter, capped at about four attempts.

---

## 🎯 Concept 1: Reliability Math: Why Dependencies Erode Uptime

### Reliability Math: Why Dependencies Erode Uptime

Availabilities of services in series multiply DOWN, so every dependency lowers your uptime. A parallel fallback multiplies it UP. This is why fallback is not optional.

Before any pattern, the math that motivates all of them. Availability is measured in **nines** - 99.9 percent is “three-nines.” The catch: when services are wired **in series** (your app needs the LLM API *and* the vector DB *and* auth, all working), their availabilities **multiply**: three 99.9 percent dependencies give `0.999 × 0.999 × 0.999 ≈ 99.7 percent` - *worse* than any single one. Every dependency you add **erodes** your uptime. The escape is **parallel** redundancy: with a fallback, you are down only if *both* the primary and the fallback fail, so `1 − (1 − 0.99)² = 99.99 percent` - a 99 percent primary plus a 99 percent fallback earns a whole extra nine. The **error budget** is the downtime a service-level objective (an **SLO**, the reliability target you promise) allows (three-nines is about 8.8 hours a year). The block computes all of this, keyless.

> 🔗 **Analogy**
>
> A chain is only as strong as its **weakest link** - and a chain of dependencies is actually *weaker* than its weakest link, because every link adds another chance to break. That is series wiring. The fix is a **spare tyre**: your car does not strand you on the first puncture, because a second, independent tyre is ready. A fallback is that spare - you are only stuck when the primary *and* the spare both fail at once, which is far rarer than either failing alone.

Your app calls three services in a chain, each 99.9 percent available. Is your app more or less available than 99.9 percent?

**📝 `01_reliability_math.py`** - *runnable*

In [ ]:
# RELIABILITY MATH - dependencies multiply your uptime DOWN, redundancy multiplies it UP.
# Availability in "nines" (99.9% = three-nines). This is the whole reason fallback exists. No key.

def nines(a): return f"{a*100:.4f}% ({str(a*100).count('9')}-nines approx)"

# SERIES: a chain where EVERY dependency must work -> availabilities MULTIPLY.
deps = {"LLM API": 0.999, "vector DB": 0.999, "auth": 0.999}
series = 1.0
for a in deps.values(): series *= a
print("A chain of dependencies (all must work) multiplies DOWN:")
for name, a in deps.items(): print(f"  {name:10} {a*100:.1f}%")
print(f"  => composite {series*100:.2f}%  (WORSE than any single one: more deps = less uptime)")

# PARALLEL: a fallback where EITHER can serve -> only both-down takes you down.
primary, fallback = 0.99, 0.99
parallel = 1 - (1 - primary) * (1 - fallback)
print(f"\nA 99% primary ALONE:                     {primary*100:.2f}%")
print(f"A 99% primary + a 99% fallback (parallel): {parallel*100:.2f}%  (a second 9 for free)")

# ERROR BUDGET: the downtime an SLO allows.
HOURS_YEAR = 365.25 * 24
for a in (0.999, 0.9999):
    down_h = (1 - a) * HOURS_YEAR
    print(f"\nSLO {a*100:.2f}% -> error budget {down_h:.2f} h/year ({down_h*60/12:.0f} min/month)")
print("\nTakeaway: every dependency erodes uptime; a fallback (parallel) is the only thing that buys it back.")
# Output:
# A chain of dependencies (all must work) multiplies DOWN:
#   LLM API    99.9%
#   vector DB  99.9%
#   auth       99.9%
#   => composite 99.70%  (WORSE than any single one: more deps = less uptime)
#
# A 99% primary ALONE:                     99.00%
# A 99% primary + a 99% fallback (parallel): 99.99%  (a second 9 for free)
#
# SLO 99.90% -> error budget 8.77 h/year (44 min/month)
#
# SLO 99.99% -> error budget 0.88 h/year (4 min/month)
#
# Takeaway: every dependency erodes uptime; a fallback (parallel) is the only thing that buys it back.

- The three 99.9 percent dependencies **in series multiply to ~99.70 percent** - worse than any one of them.
- A lone 99 percent primary is 99 percent; adding a 99 percent fallback **in parallel jumps it to 99.99 percent** - a free extra nine.
- The **error budget** falls from ~8.8 hours/year (three-nines) to ~53 minutes/year (four-nines) as you add that nine.
- The lesson every later pattern serves: dependencies erode uptime, and only redundancy (parallel) buys it back.

#### 💡 What just happened

⚡ What just happened?Reliability math: series dependencies multiply availability DOWN (more deps = less uptime), while a parallel fallback multiplies it UP. Tradeoff: redundancy costs a second provider and the code to switch, but it is the only thing that mathematically raises your uptime above your weakest dependency. The error budget is what you will monitor in Lesson 12.8.

- Chain dependencies in series and watch availability multiply DOWN
- Add a parallel fallback and watch it multiply UP; a nines / error-budget meter

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Timeouts and Deadlines: Bound Every Wait

### Timeouts and Deadlines: Bound Every Wait

A call with no timeout can hang forever and tie up a worker. A per-call timeout caps one call; a deadline budget caps the whole request across hops.

The simplest reliability bug is the one everyone ships first: a call with **no timeout**. An LLM generation that normally takes two seconds can, on a bad day, hang for minutes - and while it hangs it holds a worker, so a handful of hung calls can take your whole app down without anything actually crashing. The fix is two layers. A **per-call timeout** caps a single call, freeing the worker when it is exceeded. And across a multi-hop request, a **deadline budget** - the total time the whole request may take - is **decremented at each hop**, so a slow early dependency leaves less for the later ones and the request **aborts before it overruns** (better a fast, clear failure than a client that gives up on a half-finished request). The block shows a timeout aborting a hang and a budget flowing across three hops.

> ⏳ **Analogy**
>
> It is a **kitchen timer you never skip**. A good cook does not put a dish in the oven and wander off indefinitely - they set a timer, and when it rings they act, whether the dish is done or not. A per-call timeout is that timer for one dish. The **deadline budget** is the timer for the *whole dinner*: if the starter runs long, there is less time for the main, and at some point you serve what you have rather than keep everyone waiting past the point they will stay.

A 3-hop request shares a deadline budget of about 900ms. Hop 1 uses roughly 300ms and hop 2 roughly 400ms. How much is left for hop 3?

**📝 `02_timeouts.py`** - *runnable*

In [ ]:
# TIMEOUTS & DEADLINES - never wait forever. A per-call timeout caps one call; a DEADLINE BUDGET
# caps the whole request across hops, so a slow early dependency cannot starve the later ones.
# A tick clock (ms) stands in for real time; deterministic, no key.

def call_with_timeout(name, work_ms, timeout_ms):
    if work_ms > timeout_ms:
        return None, f"TIMEOUT after {timeout_ms}ms (the call needed {work_ms}ms)"
    return f"{name} ok", f"done in {work_ms}ms"

# A single slow call against a 2000ms timeout:
_, msg = call_with_timeout("slow-llm", work_ms=5000, timeout_ms=2000)
print("Single call with a timeout:")
print(f"  {msg}  -> the worker is freed instead of hanging forever")

# A DEADLINE BUDGET of 900ms shared across a 3-hop chain:
print("\nDeadline budget 900ms across a 3-hop chain:")
budget = 900
for hop, cost in [("retrieve", 300), ("rerank", 400), ("generate", 500)]:
    if cost > budget:
        print(f"  {hop:9} needs {cost}ms but only {budget}ms left -> ABORT (fail fast, do not overrun)")
        break
    budget -= cost
    print(f"  {hop:9} took {cost}ms, {budget}ms left")
print("\nTakeaway: bound every wait; a deadline budget stops one slow hop from blowing the whole request.")
# Output:
# Single call with a timeout:
#   TIMEOUT after 2000ms (the call needed 5000ms)  -> the worker is freed instead of hanging forever
#
# Deadline budget 900ms across a 3-hop chain:
#   retrieve  took 300ms, 600ms left
#   rerank    took 400ms, 200ms left
#   generate  needs 500ms but only 200ms left -> ABORT (fail fast, do not overrun)
#
# Takeaway: bound every wait; a deadline budget stops one slow hop from blowing the whole request.

- A call that needs far longer than its timeout is **aborted** - the worker is freed instead of hanging forever.
- The **deadline budget** is decremented across the chain: each hop’s cost is subtracted from what remains.
- When a later hop needs more time than the budget has left, it **aborts** rather than overrun the request’s total.
- A fast, clear failure beats a slow, half-finished response the client has already abandoned.

#### 💡 What just happened

⚡ What just happened?Timeouts bound a single call; a deadline budget bounds the whole request across hops. Tradeoff: too tight a timeout aborts calls that would have succeeded, too loose lets one slow dependency tie up workers - so you set them from your latency percentiles, not by guessing. One nuance for a **streaming** LLM call: a single wall-clock timeout is the wrong instrument for a token stream (too tight it kills a long-but-healthy generation, too loose it misses a silent stall), so you use an **inter-token (idle) timeout** plus a **time-to-first-token** timeout instead. Never use an infinite or default timeout on an LLM call.

- A call races a timeout bar; a slow call is aborted and the worker freed
- A deadline budget shrinks across a 3-hop chain and cuts off the last hop

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Retry with Exponential Backoff and Full Jitter

### Retry with Exponential Backoff and Full Jitter

Retry only transient errors, backing off exponentially and adding jitter so clients do not synchronize into a thundering herd. Four attempts is the sweet spot.

Many failures are **transient** - a rate limit, a brief server error, a network blip - and a retry a moment later just works. But naive retries make things worse in two ways. First, **retry the wrong errors** and you waste attempts: a 401 (auth), a 400 (bad request), or a context-overflow will fail every time, so only retry **transient** statuses (429, 500, 502, 503, 504, timeouts). Second, **retry without jitter** and you create a **thundering herd**: every client that failed at the same instant retries at the same instant, re-crushing the service just as it recovers. The fix is **exponential backoff with full jitter**: each wait roughly doubles (1s, 2s, 4s, 8s, capped) and you sleep a *random* amount within that window - `random(0, min(cap, base × 2ⁿ))` - which spreads the retries into a steady trickle. Cap it at about **four attempts**; if you need more, the upstream is broken and a circuit breaker is the right tool. One more trap: **retries stacked at multiple layers of a call chain multiply** - a 4-attempt client behind a 4-attempt gateway can drive **16x** load, a retry storm that backoff and jitter cannot fix - so retry at exactly **one** layer (the outermost caller) and cap the aggregate with a **retry budget**. The block runs a jittered retry and contrasts the herd.

> ✊ **Analogy**
>
> It is **knocking on a busy door**. If no one answers, you do not hammer the door continuously - you wait a moment and knock again, then wait *longer*, giving whoever is inside time to reach it. That is exponential backoff. And if a whole crowd is waiting at the same door, you do not *all* knock again at the exact same second (that just overwhelms the doorway) - each person waits a slightly random amount, so the knocks arrive as a manageable trickle. That randomness is jitter.

Which of these should you retry: a 503 Service Unavailable, a 401 Unauthorized, or a 400 Bad Request?

**📝 `03_retry.py`** - *runnable*

In [ ]:
# RETRY with EXPONENTIAL BACKOFF + FULL JITTER - recover transient failures without a thundering herd.
# Retry ONLY transient errors; back off exponentially; add full jitter so clients do not synchronize.
# Jitter is random(0, window); we script the fractions here so the demo is deterministic. No key.

TRANSIENT = {429, 500, 502, 503, 504}          # rate limit + server errors + timeouts -> retry
def should_retry(status): return status in TRANSIENT   # 401/403/400/context-overflow -> do NOT retry

BASE, CAP, MAX_ATTEMPTS = 1.0, 8.0, 4          # 4 total attempts is the LLM sweet spot
jitter_frac = [0.5, 0.3, 0.8]                   # illustrative "random(0,1)" picks (deterministic here)
# The call returns 503 twice, then succeeds on the 3rd attempt.
statuses = [503, 503, 200]
print("Retry a transient failure with full jitter (sleep = random(0, min(cap, base*2^n))):")
for attempt in range(MAX_ATTEMPTS):
    status = statuses[attempt] if attempt < len(statuses) else 200
    if status == 200:
        print(f"  attempt {attempt+1}: 200 OK -> success"); break
    if not should_retry(status):
        print(f"  attempt {attempt+1}: {status} non-transient -> STOP (retrying would waste time)"); break
    window = min(CAP, BASE * 2**attempt)
    sleep = round(jitter_frac[attempt] * window, 2)   # full jitter picks inside [0, window]
    print(f"  attempt {attempt+1}: {status} transient -> backoff window {window:.0f}s, jittered sleep {sleep}s")

# A non-transient error is not retried:
print("\nA 401 (auth) is NOT retried:")
print(f"  should_retry(401) = {should_retry(401)}  -> fail fast, do not burn 3 more attempts")

# A 429 can carry a Retry-After header - the SERVER tells you exactly how long to wait.
print("\nRetry-After overrides your computed backoff:")
computed = round(jitter_frac[0] * min(CAP, BASE * 2**0), 2)
retry_after = 3.0                                    # seconds, read from the 429 response header
print(f"  your computed jittered backoff: {computed}s")
print(f"  the 429 Retry-After header:      {retry_after}s -> obey the server, wait {max(retry_after, computed)}s")

# Thundering herd: without jitter, all clients retry at the SAME instant.
print("\nThundering herd (5 clients fail together, backoff window 4s):")
print("  NO jitter -> all 5 retry at exactly t=4s (a synchronized spike re-crushes the service)")
print("  FULL jitter -> the 5 spread across [0s, 4s] (a steady trickle the service can absorb)")
# Output:
# Retry a transient failure with full jitter (sleep = random(0, min(cap, base*2^n))):
#   attempt 1: 503 transient -> backoff window 1s, jittered sleep 0.5s
#   attempt 2: 503 transient -> backoff window 2s, jittered sleep 0.6s
#   attempt 3: 200 OK -> success
#
# A 401 (auth) is NOT retried:
#   should_retry(401) = False  -> fail fast, do not burn 3 more attempts
#
# Retry-After overrides your computed backoff:
#   your computed jittered backoff: 0.5s
#   the 429 Retry-After header:      3.0s -> obey the server, wait 3.0s
#
# Thundering herd (5 clients fail together, backoff window 4s):
#   NO jitter -> all 5 retry at exactly t=4s (a synchronized spike re-crushes the service)
#   FULL jitter -> the 5 spread across [0s, 4s] (a steady trickle the service can absorb)

- Two 503s are retried with **exponential backoff windows (1s, 2s)** and a **jittered sleep inside each**, then attempt 3 succeeds.
- A **401 is not retried** - `should_retry(401)` is False, so the client fails fast instead of burning three more attempts.
- The thundering-herd contrast: without jitter all 5 clients retry at *exactly* t=4s (a synchronized spike); with full jitter they **spread across [0s, 4s]**.
- Four attempts is the practical cap - more latency past that rarely catches more failures, and points to a circuit breaker instead.

#### 💡 What just happened

⚡ What just happened?Retry only transient errors, with exponential backoff plus full jitter, capped at about four attempts. Tradeoff: retries add latency and can amplify load, so jitter (to avoid the herd) and a strict transient-only filter are not optional. Honor a Retry-After header on a 429, and hand off to a circuit breaker when retries stop helping.

- Retries space out with exponential backoff; a success lands on attempt N
- Toggle jitter off (a thundering-herd spike) vs full jitter (a smooth spread)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: The Circuit Breaker: Stop Hammering a Dead Service

### The Circuit Breaker: Stop Hammering a Dead Service

After enough failures the breaker trips OPEN and fast-fails every call without touching the dead service. A cooldown probe (half-open) closes it again on recovery.

Retries handle a *brief* outage. But when a provider is **properly down**, retrying every request just wastes time and piles load onto a service that cannot answer. The **circuit breaker** is the escalation: a per-dependency state machine. It starts **closed** (calls flow, failures are counted); after `fail_max` consecutive failures it trips **open** and **fast-fails every call without calling the dead service** (so your app stays responsive with a quick, clean error instead of a slow timeout); after a **cooldown** it goes **half-open** and lets a single probe through; a success **closes** it, a failure re-opens it. This is the pattern 12.1’s fallback needs so it does not keep trying a provider it already knows is dead. The block runs the full state machine over a tick timeline.

> ⚡ **Analogy**
>
> It is an **electrical fuse**. When a circuit draws too much current, the fuse *trips* - it cuts the power fast to protect the house, rather than letting the wiring overheat and start a fire. It stays open until things are safe, then you reset it. A circuit breaker does exactly this for a failing dependency: after too many failures it trips open and stops the calls, protecting your app from wasting time on a dead service, and it probes for recovery before closing again.

A provider fails 3 times (fail_max=3) and the breaker trips OPEN. What does the very next call do?

**📝 `04_circuit_breaker.py`** - *runnable*

In [ ]:
# THE CIRCUIT BREAKER - stop hammering a dead service. A per-dependency state machine:
# CLOSED (calls flow, count failures) -> OPEN (fast-fail after fail_max) -> HALF-OPEN (probe after
# a cooldown) -> CLOSED on success. A tick timeline drives it deterministically, no key.

class CircuitBreaker:
    def __init__(self, fail_max=3, cooldown=2):
        self.fail_max, self.cooldown = fail_max, cooldown
        self.state, self.fails, self.opened_at = "CLOSED", 0, None
    def call(self, tick, provider_up):
        if self.state == "OPEN":
            if tick - self.opened_at >= self.cooldown:
                self.state = "HALF-OPEN"                 # cooldown elapsed -> probe
            else:
                return "fast-fail (open, not calling)"
        # CLOSED or HALF-OPEN: actually attempt the call
        if provider_up:
            self.state, self.fails = "CLOSED", 0          # success closes it
            return "ok"
        self.fails += 1
        if self.fails >= self.fail_max or self.state == "HALF-OPEN":
            self.state, self.opened_at = "OPEN", tick      # trip open
            return "FAIL -> tripped OPEN"
        return "FAIL (still closed)"

cb = CircuitBreaker(fail_max=3, cooldown=2)
downtime = {0: False, 1: False, 2: False, 3: False, 4: True, 5: True}  # recovers at tick 4
print("Provider is DOWN ticks 0-3, recovers at tick 4. fail_max=3, cooldown=2:")
for tick in range(6):
    result = cb.call(tick, downtime[tick])
    print(f"  tick {tick}: state={cb.state:9} -> {result}")
print("\nTakeaway: after 3 failures the breaker OPENS and fast-fails; a cooldown probe closes it on recovery.")
# Output:
# Provider is DOWN ticks 0-3, recovers at tick 4. fail_max=3, cooldown=2:
#   tick 0: state=CLOSED    -> FAIL (still closed)
#   tick 1: state=CLOSED    -> FAIL (still closed)
#   tick 2: state=OPEN      -> FAIL -> tripped OPEN
#   tick 3: state=OPEN      -> fast-fail (open, not calling)
#   tick 4: state=CLOSED    -> ok
#   tick 5: state=CLOSED    -> ok
#
# Takeaway: after 3 failures the breaker OPENS and fast-fails; a cooldown probe closes it on recovery.

- The provider is down ticks 0-3; the first two failures leave the breaker **closed** (counting).
- The third failure at tick 2 **trips it OPEN**; at tick 3 the call **fast-fails** without touching the provider.
- After the cooldown, tick 4 is a **half-open probe** - the provider has recovered, so the probe succeeds and the breaker **closes**.
- Net effect: your app stopped hammering the dead provider and recovered automatically, all without a human at 2am.

#### 💡 What just happened

⚡ What just happened?The circuit breaker trips open after repeated failures and fast-fails until a cooldown probe finds the service recovered. Tradeoff: while open it rejects calls that might have started succeeding a moment early, but that is far cheaper than piling timeouts onto a down service. Tune fail_max and the cooldown to the dependency; pair it with the fallback (Step 5).

- A state light goes closed -> open (after fail_max) -> half-open (cooldown) -> closed
- A down/up timeline drives it; while open it fast-fails without calling

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Fallback and Graceful Degradation: Something Beats Nothing

### Fallback and Graceful Degradation: Something Beats Nothing

Try providers in priority order with health tracking; the first success wins. When all providers fail, degrade down a ladder - cache, then cheaper, then canned - so the user always gets something.

Now compose failover with a floor. **Fallback** tries providers in priority order with **health tracking** (skip one whose breaker is open), and the **first success wins** - a chain like an **OpenAI** primary, an **Anthropic** (Claude) fallback, and a **Gemini** tertiary is 12.1’s pattern, now health-aware. But what happens when *every* provider is down? The naive answer is a 500; the right answer is **graceful degradation**: return *something* useful instead of an error. The **degradation ladder** steps down in quality: the primary model, then a cheaper/smaller model, then a **cached** (possibly stale) answer, then a **canned** templated response (“We are experiencing high demand - here is our FAQ”). Each rung is worse than the last, but every rung beats a blank error page. The block runs a fallback chain and then degrades through the ladder, always returning a result.

> 🍽️ **Analogy**
>
> It is a **restaurant that is out of your first choice**. A good waiter does not just say “kitchen’s closed” and walk away - they offer the next best thing: “we are out of the salmon, but the sea bass is excellent,” and if that is gone too, “here is the soup, on the house.” Each option is a step down, but you never leave hungry. Graceful degradation is that waiter: primary model, then a cheaper one, then a cached answer, then at least a helpful canned reply - anything but an empty plate.

Every one of your LLM providers is down. What should the app return?

**📝 `05_fallback_degrade.py`** - *runnable*

In [ ]:
# FALLBACK & GRACEFUL DEGRADATION - something beats nothing. Try providers in priority order with
# health tracking; when ALL providers fail, DEGRADE down a ladder (cache -> cheaper -> canned) so the
# user always gets SOMETHING instead of an error page. Scripted health, deterministic, no key.

def serve(providers, cache_hit):
    for name, up in providers:                      # 1) health-tracked fallback chain
        if up:
            return f"answer from {name}", name
    # 2) all providers down -> degrade down the ladder
    if cache_hit:
        return "stale cached answer (a few minutes old)", "cache"
    return "canned: 'We are experiencing high demand - here is our FAQ link.'", "canned"

print("Scenario A - primary+secondary down, tertiary up:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", True)], cache_hit=False)
print(f"  served by {via}: {r}")

print("\nScenario B - ALL providers down, cache has a stale copy:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", False)], cache_hit=True)
print(f"  degraded to {via}: {r}")

print("\nScenario C - ALL providers down, cache empty:")
r, via = serve([("primary", False), ("secondary", False), ("tertiary", False)], cache_hit=False)
print(f"  degraded to {via}: {r}")

print("\nTakeaway: a health-tracked chain, then a degradation ladder - the user ALWAYS gets a response.")
# Output:
# Scenario A - primary+secondary down, tertiary up:
#   served by tertiary: answer from tertiary
#
# Scenario B - ALL providers down, cache has a stale copy:
#   degraded to cache: stale cached answer (a few minutes old)
#
# Scenario C - ALL providers down, cache empty:
#   degraded to canned: canned: 'We are experiencing high demand - here is our FAQ link.'
#
# Takeaway: a health-tracked chain, then a degradation ladder - the user ALWAYS gets a response.

- Scenario A: the primary and secondary are down, so the **health-tracked chain falls through to the tertiary**, which serves the answer.
- Scenario B: *all* providers are down but the **cache has a stale copy** - degrade to that rather than error.
- Scenario C: all providers down and the cache is empty - degrade to a **canned response** with an FAQ link.
- In every scenario the function **returns something**: the degradation ladder guarantees the user is never left with a blank error.

#### 💡 What just happened

⚡ What just happened?Fallback tries providers by health (first success wins); when all fail, a degradation ladder (cache -> cheaper -> canned) still returns something. Tradeoff: a degraded answer is lower quality and you must label it honestly (a stale-cache banner, a ‘high demand’ notice), but a useful-if-imperfect response beats a 500. The cache rung becomes first-class in Lesson 12.4.

- A request falls through a provider chain by health; first success wins
- When all fail it degrades down a ladder (cache -> cheaper -> canned) and ALWAYS returns

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Bulkheads and Idempotent Retries: Contain and Deduplicate

### Bulkheads and Idempotent Retries: Contain and Deduplicate

A bulkhead bounds concurrency per dependency so one failing dependency cannot sink the whole app. And because you now retry, retried writes must be idempotent or they run twice.

Two patterns that make the rest safe. First, the **bulkhead**: give each dependency its own **bounded concurrency pool**, so if one dependency gets slow or floods with requests, it can only fill *its own* pool - the workers serving your other dependencies are untouched. Without bulkheads, one slow dependency consumes every worker and the whole app hangs; with them, the failure is **contained to one compartment**. Second, **idempotency**: because Step 3 now *retries*, any retried **write** (charge a card, send an email, insert a row) will execute *twice* unless it carries an **idempotency key** that makes the second delivery a no-op. Retries and idempotency are a package deal - you cannot safely add one without the other. The block runs a bulkhead isolating a flood and a keyed write deduping a retry.

> 🚢 **Analogy**
>
> The bulkhead is literally named after a **ship’s watertight compartments**. A ship’s hull is divided into sealed sections so that if one is breached and floods, the water is *contained* - the other compartments stay dry and the ship stays afloat. Without those bulkheads, a single hole sinks the whole vessel. Isolating each dependency in its own resource pool is exactly this: one flooded compartment cannot take down the ship.

Six requests flood a slow dependency that has a 2-slot bulkhead. What happens to the other four?

**📝 `06_bulkhead_idem.py`** - *runnable*

In [ ]:
# BULKHEADS & IDEMPOTENT RETRIES - contain failures, and make retries safe. A BULKHEAD bounds the
# concurrency PER dependency (like a ship's watertight compartments) so one slow/failing dependency
# cannot consume every worker. And a retried write must be IDEMPOTENT or it runs twice. No key.

# --- Bulkhead: a slow dependency floods, but only its own 2-slot pool ---
def admit(calls, pool_size):
    running, rejected = [], []
    for c in calls:
        (running if len(running) < pool_size else rejected).append(c)
    return running, rejected

running, rejected = admit([f"slow-{i}" for i in range(6)], pool_size=2)
print("Bulkhead - 6 calls hit a SLOW dependency with a 2-slot pool:")
print(f"  running (in its pool): {running}")
print(f"  rejected/shed:         {rejected}  <- the fast dependency's pool is UNTOUCHED (isolated)")

# --- Idempotency: a retried write executes once with a key, twice without ---
charges, seen = [], set()
def charge_naive(order):      charges.append(order)                       # no key -> double-charges
def charge_keyed(order, key):
    if key in seen: return "duplicate ignored"
    seen.add(key); charges.append(order); return "charged"

charge_naive("A"); charge_naive("A")                                     # a retry delivers it twice
naive_count = charges.count("A")
charge_keyed("B", "idem-1"); charge_keyed("B", "idem-1")                 # same key twice
print("\nIdempotency - the SAME write delivered twice (a retry):")
print(f"  naive (no key): order A charged {naive_count} times  <- BUG: the retry double-charged")
print(f"  keyed:          order B charged {charges.count('B')} time   <- the key made the retry a no-op")
print("\nTakeaway: bulkheads isolate a failure to one compartment; idempotency keys make retries safe.")
# Output:
# Bulkhead - 6 calls hit a SLOW dependency with a 2-slot pool:
#   running (in its pool): ['slow-0', 'slow-1']
#   rejected/shed:         ['slow-2', 'slow-3', 'slow-4', 'slow-5']  <- the fast dependency's pool is UNTOUCHED (isolated)
#
# Idempotency - the SAME write delivered twice (a retry):
#   naive (no key): order A charged 2 times  <- BUG: the retry double-charged
#   keyed:          order B charged 1 time   <- the key made the retry a no-op
#
# Takeaway: bulkheads isolate a failure to one compartment; idempotency keys make retries safe.

- The bulkhead admits **2 of the 6 flooding calls** into the slow dependency’s pool and sheds the other 4 - a fast dependency’s pool is **untouched**.
- That isolation is the point: one slow dependency cannot consume every worker and sink the whole app.
- The idempotency demo: a naive write **charges order A twice** (the retry double-acted); a **keyed write charges order B once** (the key made the retry a no-op).
- Because you now retry (Step 3), every retried write MUST be idempotent - the two patterns are a package deal.

#### 💡 What just happened

⚡ What just happened?Bulkheads isolate a failure to one compartment (a bounded pool per dependency); idempotency keys make retried writes safe. Tradeoff: bulkheads can shed load you might have served, and idempotency keys add bookkeeping, but together they stop one slow dependency from cascading and stop retries from double-acting. Idempotency internals are Lesson 11.9.

- A flood fills one dependency's bounded pool while others stay free (isolation)
- A retried keyed write dedupes to one vs a naive double-execute

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Resilient Client: All Patterns, One Wrapper

### The Resilient Client: All Patterns, One Wrapper

Compose timeout, circuit breaker, retry, fallback, and degradation into one client - and get the order right, so an open breaker fast-fails instead of wasting retries.

Now assemble everything into one **resilient client**, because in production these patterns are not separate - they wrap each other, and the **order matters**. A single call is bounded by a **timeout**; a per-provider **circuit breaker** gates it, so a provider whose breaker is **open** is skipped instantly without wasting a single retry; **retry with backoff** re-attempts transient failures against a healthy provider; **fallback** moves to the next provider when one is exhausted or open; and **degradation** is the floor when everything fails. The breaker sitting *in front of* the retry is the key ordering insight: you never want to burn four retries on a provider you already know is down. The block traces one request through all the layers - skipping an open breaker, surviving a transient failure via retry, and landing on a fallback.

> 🚗 **Analogy**
>
> It is a **car’s safety systems working together**. A modern car does not rely on one safeguard - it layers them: anti-lock brakes, traction control, airbags, a seatbelt, a crumple zone. Each handles a different failure, and they are *ordered* (the seatbelt holds you before the airbag fires). The resilient client is that stack for a network call: timeout, breaker, retry, fallback, and degradation each handle a different failure mode, composed in the right order so they reinforce rather than fight each other.

In the composed client, provider A’s breaker is OPEN. What does the client do first?

**📝 `07_resilient_client.py`** - *runnable*

In [ ]:
# THE RESILIENT CLIENT - all patterns, one wrapper, and the ORDER matters. A call is bounded by a
# TIMEOUT, gated by a per-provider CIRCUIT BREAKER, re-attempted by RETRY, moved on by FALLBACK, and
# floored by DEGRADATION. We trace one request through every layer. Scripted, deterministic, no key.

def resilient_call(providers, cache_hit):
    trace = []
    for name, breaker_open, attempts in providers:     # FALLBACK: try each provider in order
        if breaker_open:                                # CIRCUIT BREAKER: skip a provider that is open
            trace.append(f"{name}: breaker OPEN -> fast-fail, skip (no retries wasted)")
            continue
        for i, ok in enumerate(attempts):               # RETRY (each attempt is TIMEOUT-bounded)
            if ok:
                trace.append(f"{name}: attempt {i+1} within timeout -> OK")
                return "answer from " + name, trace
            trace.append(f"{name}: attempt {i+1} transient fail -> retry with backoff")
    # DEGRADATION: everything failed -> the floor
    trace.append("all providers exhausted -> degrade to " + ("cache" if cache_hit else "canned response"))
    return ("stale cache" if cache_hit else "canned response"), trace

# provider A breaker is OPEN; provider B fails once then succeeds on retry.
result, trace = resilient_call(
    providers=[("A", True, []), ("B", False, [False, True])],
    cache_hit=False)
print("One request through the resilient client (order: timeout -> breaker -> retry -> fallback -> degrade):")
for line in trace: print(f"  {line}")
print(f"  RESULT: {result}")
print("\nTakeaway: composed layers turn a fragile call into one that survives an open breaker AND a transient fail.")
# Output:
# One request through the resilient client (order: timeout -> breaker -> retry -> fallback -> degrade):
#   A: breaker OPEN -> fast-fail, skip (no retries wasted)
#   B: attempt 1 transient fail -> retry with backoff
#   B: attempt 2 within timeout -> OK
#   RESULT: answer from B
#
# Takeaway: composed layers turn a fragile call into one that survives an open breaker AND a transient fail.

- Provider A’s breaker is **OPEN**, so the client **fast-fails and skips A** - no retries wasted on a known-dead provider.
- It falls back to provider B; B’s first attempt is a transient failure, so **retry with backoff** kicks in.
- B’s second attempt succeeds **within the timeout**, and the request returns an answer from B.
- The layer order - timeout, breaker, retry, fallback, degrade - is what lets one request survive both an open breaker AND a transient failure.

**📝 `07b_tenacity_pybreaker.py`** - *illustrative*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


- `pybreaker.CircuitBreaker(fail_max=3, reset_timeout=30)` is the real circuit breaker - Step 4 as a library.
- `tenacity`’s `@retry` gives `stop_after_attempt(4)`, `wait_random_exponential` (which is AWS full jitter: a random wait in [0, the exponential cap]), and `retry_if_exception_type` (transient-only) - Step 3 in three lines.
- `@breaker` wraps `@retry` so an **open breaker fast-fails** before any retry runs - the Step 7 ordering, enforced by decorator order.
- **pyresilience** unifies retry + breaker + timeout + fallback behind one `@resilient()` decorator with a 429-aware `llm_policy()` that honors `Retry-After`.

#### 💡 What just happened

⚡ What just happened?The resilient client composes timeout, circuit breaker, retry, fallback, and degradation in one wrapper, ordered so an open breaker fast-fails before retrying. Tradeoff / the whole lesson: each layer adds a little code and latency on the failure path, but together they turn a fragile call that dies with its provider into one that survives outages, rate limits, and slow dependencies. In production you compose libraries (tenacity, pybreaker, pyresilience), not hand-rolled loops.

- One request threads timeout -> breaker -> retry -> fallback -> degrade
- A provider's breaker is open, the next fails once then succeeds; layers light in order

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Reliability starts from a hard truth in the **math**: every dependency multiplies your uptime *down*, and only a parallel **fallback** multiplies it back up (Step 1). So you defend the call in depth. A **timeout** and a **deadline budget** stop any wait from hanging the app (Step 2). **Retry with backoff and full jitter** recovers transient failures without a thundering herd (Step 3). A **circuit breaker** escalates to fast-fail when a provider is properly down (Step 4). **Fallback with graceful degradation** keeps returning *something* even when everything fails (Step 5). **Bulkheads** contain a failure to one compartment and **idempotency** makes the retries safe (Step 6). And you compose all of it into one **resilient client**, ordered so an open breaker never wastes a retry (Step 7). That is how an app stops dying with its provider at 2am.

| Pattern | Protects against | Mechanism | Cost / tradeoff |
|---|---|---|---|
| Timeout + deadline | a hung / slow dependency | cap each call; a budget across hops | too tight aborts good calls |
| Retry + jitter | transient failures (429/5xx) | exponential backoff + full jitter | latency; needs idempotency |
| Circuit breaker | a provider that is properly down | fast-fail while open; probe to recover | may reject a just-recovered call |
| Fallback + degrade | a provider (or all) failing | health-tracked chain; degradation ladder | degraded answers are lower quality |
| Bulkhead | one dependency sinking the app | bounded pool per dependency | sheds load under a flood |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now keep an AI app standing when its dependencies fail. These same retries and fallbacks become gateway configuration in Lesson 12.3, caching becomes a first-class layer (and the top rung of the degradation ladder) in Lesson 12.4, autoscaling and load shedding are in Lesson 12.7, and monitoring the error budget with SLO alerts is in Lesson 12.8 (with incident runbooks in Lesson 12.9). The primary references are the [AWS Builders’ Library on timeouts, retries, and backoff with jitter](https://aws.amazon.com/builders-library/timeouts-retries-and-backoff-with-jitter/), the [ACM “Calculus of Service Availability”](https://queue.acm.org/detail.cfm?id=3096459), and the [pyresilience](https://github.com/AhsanSheraz/pyresilience), [tenacity](https://github.com/jd/tenacity), and [pybreaker](https://github.com/danielfm/pybreaker) libraries.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Designingfor Reliability**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.2.html` - regenerate this notebook whenever the source HTML is updated.*
